In [1]:
from google.colab import drive
drive.mount('/content/drive')  # Mount Google Drive
import os

# Define dataset and model paths
drive_path = '/content/drive/My Drive/'  # Adjust this to your Google Drive path

Mounted at /content/drive


In [2]:
# ============================================================
# UrbanSound8K 10-Class Training (Librosa + MLP) - Spyder Friendly
# ============================================================
# ✅ Multi-feature extraction using librosa
# ✅ MLPClassifier training
# ✅ Prints accuracy, classification report, confusion matrix
# ✅ Saves confusion matrix images + report text
# ✅ Saves model + meta (joblib) for Tkinter testing
#
# Install:
#   pip install numpy pandas librosa soundfile scikit-learn joblib matplotlib
#
# Dataset structure (typical):
#   UrbanSound8K.csv
#   audio/
#     fold1/xxx.wav
#     fold2/xxx.wav
#     ...
#     fold10/xxx.wav
#
# Run in Spyder: just press F5 after setting paths below.
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ===========================
# ✅ EDIT THESE PATHS
# ===========================
CSV_PATH   = r"/content/drive/MyDrive/metadata/UrbanSound8K.csv"
AUDIO_DIR  = r"/content/drive/MyDrive/audio"     # contains fold1..fold10
MODEL_DIR  = r"/content/drive/MyDrive/metadata/models_10class"

# ===========================
# Training Settings
# ===========================
TEST_FOLD = 10
SEED = 42

SR = 22050
DURATION = 4.0  # UrbanSound8K clips are <= 4 seconds (safe)
HOP = 512

# Augmentation (train only)
AUGMENT = True
AUG_PER_FILE = 2   # try 0/2/4

# Balance train set by oversampling (useful if your data is uneven)
BALANCE = True

# PCA often helps MLP on many features (set 0 to disable)
PCA_COMPONENTS = 256   # try 0 / 128 / 256

# MLP settings
HIDDEN = (256, 128)
MAX_ITER = 500


# ============================================================
# Helpers
# ============================================================
def fix_length(y, target_len):
    if len(y) < target_len:
        return np.pad(y, (0, target_len - len(y)), mode="constant")
    return y[:target_len]

def find_audio_path(audio_root: Path, fold: int, fname: str) -> Path:
    # UrbanSound8K default: audio/fold{fold}/file.wav
    p1 = audio_root / f"fold{int(fold)}" / fname
    if p1.exists():
        return p1
    # fallback: audio/file.wav
    p2 = audio_root / fname
    if p2.exists():
        return p2
    return None

def augment_audio(y, sr, rng):
    # simple + safe augmentation
    y2 = y.astype(np.float32).copy()
    y2 *= rng.uniform(0.8, 1.2)  # gain
    shift = rng.randint(-int(0.12 * sr), int(0.12 * sr))  # time shift
    y2 = np.roll(y2, shift)
    noise_amp = rng.uniform(0.0005, 0.004)               # noise
    y2 += noise_amp * rng.randn(len(y2)).astype(np.float32)
    return y2

def extract_features(y, sr, n_mfcc=40, n_mels=128, hop_length=512):
    """
    Multi-feature fixed vector using mean+std pooling:
    - MFCC (40) mean+std
    - Chroma (12) mean+std
    - Log-mel (128) mean+std
    - Spectral contrast (~7) mean+std
    - Tonnetz (6) mean+std
    - centroid/bandwidth/rolloff/zcr/rms mean+std
    """
    y = librosa.util.normalize(y.astype(np.float32))

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, hop_length=hop_length)
    mfcc_mean, mfcc_std = mfcc.mean(axis=1), mfcc.std(axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length)
    chroma_mean, chroma_std = chroma.mean(axis=1), chroma.std(axis=1)

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=hop_length, power=2.0)
    logmel = librosa.power_to_db(mel, ref=np.max)
    logmel_mean, logmel_std = logmel.mean(axis=1), logmel.std(axis=1)

    try:
        contrast = librosa.feature.spectral_contrast(S=mel, sr=sr)
    except Exception:
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    contrast_mean, contrast_std = contrast.mean(axis=1), contrast.std(axis=1)

    try:
        y_harm = librosa.effects.harmonic(y)
        tonnetz = librosa.feature.tonnetz(y=y_harm, sr=sr)
        tonnetz_mean, tonnetz_std = tonnetz.mean(axis=1), tonnetz.std(axis=1)
    except Exception:
        tonnetz_mean = np.zeros((6,), dtype=np.float32)
        tonnetz_std = np.zeros((6,), dtype=np.float32)

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop_length)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop_length)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop_length)
    zcr = librosa.feature.zero_crossing_rate(y=y, hop_length=hop_length)
    rms = librosa.feature.rms(y=y, hop_length=hop_length)

    summary = np.array([
        centroid.mean(), centroid.std(),
        bandwidth.mean(), bandwidth.std(),
        rolloff.mean(), rolloff.std(),
        zcr.mean(), zcr.std(),
        rms.mean(), rms.std()
    ], dtype=np.float32)

    feat = np.concatenate([
        mfcc_mean, mfcc_std,
        chroma_mean, chroma_std,
        logmel_mean, logmel_std,
        contrast_mean, contrast_std,
        tonnetz_mean, tonnetz_std,
        summary
    ], axis=0).astype(np.float32)

    return np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def oversample_balance(X, y, seed=42):
    rng = np.random.RandomState(seed)
    classes, counts = np.unique(y, return_counts=True)
    max_count = int(counts.max())
    idx_all = []
    for c, cnt in zip(classes, counts):
        idx_c = np.where(y == c)[0]
        if cnt < max_count:
            extra = rng.choice(idx_c, size=(max_count - cnt), replace=True)
            idx_c = np.concatenate([idx_c, extra])
        idx_all.append(idx_c)
    idx_all = np.concatenate(idx_all)
    rng.shuffle(idx_all)
    return X[idx_all], y[idx_all]

def plot_confusion(cm, labels, out_path, normalize=False):
    if normalize:
        cm_plot = cm.astype(np.float32)
        cm_plot = cm_plot / (cm_plot.sum(axis=1, keepdims=True) + 1e-9)
    else:
        cm_plot = cm.astype(np.int64)

    fig = plt.figure(figsize=(10, 8))
    ax = plt.gca()
    im = ax.imshow(cm_plot, interpolation="nearest")
    ax.set_title("Confusion Matrix" + (" (Normalized)" if normalize else ""))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    # show numbers only for <= 12 classes (we have 10 -> ok)
    fmt = ".2f" if normalize else "d"
    thresh = float(cm_plot.max()) * 0.6 if cm_plot.size else 0.0
    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            val = cm_plot[i, j]
            txt = format(float(val), fmt) if normalize else format(int(val), fmt)
            ax.text(j, i, txt, ha="center", va="center",
                    color="white" if float(val) > thresh else "black")

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Build dataset
# ============================================================
def build_xy(df, audio_root, sr, duration, augment=False, aug_k=0, seed=42):
    rng = np.random.RandomState(seed)
    target_len = int(sr * duration)

    X_list, y_list = [], []
    missing = 0

    for i in range(len(df)):
        fname = str(df.loc[i, "slice_file_name"])
        fold = int(df.loc[i, "fold"])
        path = find_audio_path(audio_root, fold, fname)

        if path is None:
            missing += 1
            continue

        try:
            y, _ = librosa.load(str(path), sr=sr, mono=True)
        except Exception:
            missing += 1
            continue

        if y is None or len(y) == 0:
            missing += 1
            continue

        y = fix_length(y.astype(np.float32), target_len)
        X_list.append(extract_features(y, sr=sr, hop_length=HOP))
        y_list.append(int(df.loc[i, "classID"]))  # 0..9

        if augment and aug_k > 0:
            for _k in range(aug_k):
                y_aug = augment_audio(y, sr=sr, rng=rng)
                y_aug = fix_length(y_aug, target_len)
                X_list.append(extract_features(y_aug, sr=sr, hop_length=HOP))
                y_list.append(int(df.loc[i, "classID"]))

    if missing:
        print(f"[WARN] Missing/failed audio files: {missing} (check AUDIO_DIR structure)")

    X = np.vstack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.int64)
    return X, y


# ============================================================
# Train
# ============================================================
def main():
    csv_path = Path(CSV_PATH)
    audio_root = Path(AUDIO_DIR)
    model_dir = Path(MODEL_DIR)
    reports_dir = model_dir / "reports"
    model_dir.mkdir(parents=True, exist_ok=True)
    reports_dir.mkdir(parents=True, exist_ok=True)

    print("===================================")
    print("CWD       :", os.getcwd())
    print("CSV       :", csv_path.resolve())
    print("AUDIO_DIR :", audio_root.resolve())
    print("MODEL_DIR :", model_dir.resolve())
    print("===================================")

    df = pd.read_csv(csv_path)

    required = {"slice_file_name", "fold", "classID"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"CSV missing required columns. Need: {required}. Found: {set(df.columns)}")

    # get class names in correct order 0..9
    if "class" in df.columns:
        class_map = df[["classID", "class"]].drop_duplicates().sort_values("classID")
        labels = class_map["class"].astype(str).tolist()
    else:
        labels = [str(i) for i in range(10)]

    # split by fold
    train_df = df[df["fold"] != int(TEST_FOLD)].reset_index(drop=True)
    test_df  = df[df["fold"] == int(TEST_FOLD)].reset_index(drop=True)
    print(f"[OK] Fold split: train={len(train_df)} test={len(test_df)} (test_fold={TEST_FOLD})")

    print("[1/3] Building train features...")
    X_train, y_train = build_xy(train_df, audio_root, SR, DURATION, augment=AUGMENT, aug_k=AUG_PER_FILE, seed=SEED)

    print("[2/3] Building test features...")
    X_test, y_test = build_xy(test_df, audio_root, SR, DURATION, augment=False, aug_k=0, seed=SEED)

    print(f"[OK] Shapes: X_train={X_train.shape} X_test={X_test.shape}")

    if BALANCE:
        X_train, y_train = oversample_balance(X_train, y_train, seed=SEED)
        print(f"[OK] After balance: X_train={X_train.shape}")

    steps = [("scaler", StandardScaler())]
    if PCA_COMPONENTS and PCA_COMPONENTS > 0:
        steps.append(("pca", PCA(n_components=int(PCA_COMPONENTS), random_state=SEED)))

    steps.append((
        "mlp",
        MLPClassifier(
            hidden_layer_sizes=HIDDEN,
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size=128,
            learning_rate="adaptive",
            learning_rate_init=1e-3,
            max_iter=MAX_ITER,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=SEED,
            verbose=True
        )
    ))

    clf = Pipeline(steps=steps)

    print("[3/3] Training MLP...")
    clf.fit(X_train, y_train)

    # eval
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print("\n==============================")
    print(f"Test Accuracy: {acc:.4f}")
    print("==============================\n")

    report = classification_report(y_test, y_pred, target_names=labels, digits=4)
    print(report)
    print("Confusion matrix:")
    print(cm)

    # save reports
    report_txt = reports_dir / "classification_report.txt"
    with open(report_txt, "w", encoding="utf-8") as f:
        f.write(f"UrbanSound8K 10-class MLP\n")
        f.write(f"Test fold: {TEST_FOLD}\n")
        f.write(f"Accuracy : {acc:.6f}\n\n")
        f.write(report)
        f.write("\n\nConfusion matrix:\n")
        f.write(np.array2string(cm))

    plot_confusion(cm, labels, reports_dir / "confusion_matrix.png", normalize=False)
    plot_confusion(cm, labels, reports_dir / "confusion_matrix_normalized.png", normalize=True)

    # save model + meta
    model_path = model_dir / "urbansound10_mlp_pipeline.joblib"
    meta_path  = model_dir / "urbansound10_meta.joblib"

    joblib.dump(clf, model_path)
    joblib.dump(
        {
            "task": "class10",
            "label_names": labels,      # IMPORTANT for your Tkinter UI
            "sr": SR,
            "duration": DURATION,
            "test_fold": TEST_FOLD,
            "augment": AUGMENT,
            "aug_per_file": AUG_PER_FILE,
            "balance": BALANCE,
            "pca_components": PCA_COMPONENTS,
        },
        meta_path
    )

    print(f"\n[OK] Saved model: {model_path}")
    print(f"[OK] Saved meta : {meta_path}")
    print(f"[OK] Saved reports in: {reports_dir}")


if __name__ == "__main__":
    main()

CWD       : /content
CSV       : /content/drive/MyDrive/metadata/UrbanSound8K.csv
AUDIO_DIR : /content/drive/MyDrive/audio
MODEL_DIR : /content/drive/MyDrive/metadata/models_10class
[OK] Fold split: train=7895 test=837 (test_fold=10)
[1/3] Building train features...
[WARN] Missing/failed audio files: 7894 (check AUDIO_DIR structure)
[2/3] Building test features...
[WARN] Missing/failed audio files: 837 (check AUDIO_DIR structure)


ValueError: need at least one array to concatenate